
# ДЗ 3 — Более сложная модель, подбор гиперпараметров и интерпретация

**Студентка:** Григорьева Арина Анатольевна  
**Курс:** Машинное обучение и нейросети, ИТМО, весна 2026

В этом ноутбуке строится более сложная модель для задачи бинарной классификации наличия болезни сердца. В качестве ансамблевой модели выбран `RandomForestClassifier`, для него выполняется подбор гиперпараметров с помощью `GridSearchCV` на кросс-валидации. После этого модель оценивается на отложенной выборке по тем же метрикам, которые были выбраны ранее: **ROC-AUC** как основная метрика и **Recall** как дополнительная метрика.

Дополнительно выполняется интерпретация модели:
- **глобальная** — через permutation importance;
- **локальная** — через SHAP для отдельных объектов.



## 1. Почему выбрана именно эта модель

Для третьего задания нужна модель сложнее линейного бейзлайна. Я выбрала **случайный лес** (`RandomForestClassifier`), потому что:

1. это ансамбль деревьев решений, то есть модель уже заметно богаче по выразительности, чем логистическая регрессия;
2. модель умеет учитывать **нелинейные зависимости** и **взаимодействия признаков**, что уместно для медицинских данных;
3. случайный лес устойчив к выбросам и не требует обязательного масштабирования признаков;
4. модель хорошо интерпретируется: можно анализировать важности признаков и объяснять отдельные предсказания.

В отличие от линейной модели, здесь не нужен `StandardScaler`, поскольку деревья и ансамбли деревьев не чувствительны к масштабу признаков. Признаки, которые по смыслу являются категориальными (`cp`, `restecg`, `slope`, `thal` и др.), в датасете уже закодированы целыми числами. Для деревьев такое представление допустимо как практичный baseline для небольшого датасета. Для более строгого production-подхода можно было бы дополнительно сравнить вариант с one-hot encoding.


In [ ]:

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.2

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)



## 2. Загрузка данных

Для воспроизводимости ноутбук сначала пытается прочитать локальный файл `data/dataset.csv`, а если его нет — загружает датасет по ссылке.


In [ ]:

DATA_URL = 'https://raw.githubusercontent.com/kb22/Heart-Disease-Prediction/master/dataset.csv'
LOCAL_DATA_PATH = Path('data/dataset.csv')

if LOCAL_DATA_PATH.exists():
    df = pd.read_csv(LOCAL_DATA_PATH)
else:
    df = pd.read_csv(DATA_URL)

print(f'Датасет: {df.shape[0]} строк, {df.shape[1]} столбцов')
df.head()


In [ ]:

X = df.drop(columns='target')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Размер train:', X_train.shape)
print('Размер test:', X_test.shape)
print('Доля положительного класса в train:', round(y_train.mean(), 3))
print('Доля положительного класса в test:', round(y_test.mean(), 3))



Разбиение выполнено со стратификацией по целевой переменной, чтобы доля классов в обучающей и тестовой выборках оставалась сопоставимой. Это особенно важно для медицинской задачи, где некорректное распределение классов между выборками может исказить оценку качества.



## 3. Подбор гиперпараметров

Для настройки модели используется `GridSearchCV` с 5-fold стратифицированной кросс-валидацией.

Оптимизируемые гиперпараметры:
- `n_estimators` — количество деревьев;
- `max_depth` — максимальная глубина дерева;
- `min_samples_split` — минимальное число объектов для разбиения узла;
- `min_samples_leaf` — минимальное число объектов в листе;
- `max_features` — число признаков, рассматриваемых при поиске лучшего разбиения.

Основная метрика для подбора — **ROC-AUC**, поскольку именно она была выбрана в предыдущем задании как главная метрика качества. Дополнительно при финальной оценке на тесте будет посчитан `Recall`.


In [ ]:

param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 3, 5, 7],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf = RandomForestClassifier(
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1,
)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

grid_search.fit(X_train, y_train)

print('Лучшие параметры:')
print(grid_search.best_params_)
print(f'Лучшая средняя ROC-AUC на CV: {grid_search.best_score_:.4f}')



После `GridSearchCV` лучшая модель уже автоматически переобучена на всей обучающей выборке (`refit=True`). Именно её и нужно использовать для финальной оценки на тестовой выборке.


In [ ]:

best_model = grid_search.best_estimator_

best_model



## 4. Оценка качества на отложенной выборке


In [ ]:

y_test_proba = best_model.predict_proba(X_test)[:, 1]
y_test_pred = best_model.predict(X_test)

test_roc_auc = roc_auc_score(y_test, y_test_proba)
test_recall = recall_score(y_test, y_test_pred)

print(f'Test ROC-AUC: {test_roc_auc:.4f}')
print(f'Test Recall:  {test_recall:.4f}')
print()
print(classification_report(y_test, y_test_pred))


In [ ]:

ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, cmap='Blues')
plt.title('Матрица ошибок на тестовой выборке')
plt.show()



На этом этапе важно сравнить качество ансамблевой модели с бейзлайном из предыдущего задания. Если случайный лес даёт более высокий `ROC-AUC` и при этом не теряет слишком сильно в `Recall`, то его можно считать более сильным кандидатом для данной задачи.



## 5. Глобальная интерпретация модели

Для глобальной интерпретации я использую **permutation importance**. Этот подход удобен тем, что он измеряет, насколько ухудшается качество модели, если случайно перемешать значения отдельного признака. В отличие от встроенных `feature_importances_` у деревьев, permutation importance сильнее привязана к фактическому влиянию признака на качество модели.


In [ ]:

perm_result = permutation_importance(
    estimator=best_model,
    X=X_test,
    y=y_test,
    n_repeats=30,
    random_state=RANDOM_STATE,
    scoring='roc_auc',
)

perm_importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std,
}).sort_values('importance_mean', ascending=False)

perm_importance_df


In [ ]:

plt.figure(figsize=(10, 6))
sns.barplot(
    data=perm_importance_df,
    x='importance_mean',
    y='feature',
    orient='h',
)
plt.title('Permutation importance на тестовой выборке (ROC-AUC)')
plt.xlabel('Среднее падение ROC-AUC при перемешивании признака')
plt.ylabel('Признак')
plt.show()



**Комментарий к глобальной интерпретации.**  
Ожидаемо среди наиболее важных признаков должны оказаться характеристики, которые клинически связаны с сердечно-сосудистым риском: тип боли в груди (`cp`), наличие стенокардии при нагрузке (`exang`), депрессия сегмента ST (`oldpeak`), максимальная ЧСС (`thalach`), число поражённых сосудов (`ca`), результаты теста `thal`. Если именно эти признаки поднимаются в топе важности, это делает поведение модели содержательно правдоподобным.



## 6. Локальная интерпретация отдельных предсказаний через SHAP

Для локальной интерпретации используется **SHAP**. Этот метод позволяет разложить индивидуальный прогноз модели на вклады отдельных признаков: какие признаки подталкивают вероятность болезни вверх, а какие — вниз.

Если пакет `shap` ещё не установлен, раскомментируй следующую строку и выполни её один раз:

```python
# %pip install shap
```


In [ ]:

import shap

shap.initjs()


In [ ]:

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)


In [ ]:

shap.summary_plot(shap_values[1], X_test, plot_type='bar')



График выше тоже даёт глобальную картину, но уже в терминологии SHAP. Он показывает, какие признаки в среднем вносят наибольший вклад в предсказания класса `1`.


In [ ]:

example_idx = 0

print('Индекс объекта в X_test:', X_test.index[example_idx])
print('Истинный класс:', y_test.iloc[example_idx])
print('Предсказанный класс:', y_test_pred[example_idx])
print('Вероятность класса 1:', round(y_test_proba[example_idx], 4))

X_test.iloc[[example_idx]]


In [ ]:

shap.force_plot(
    base_value=explainer.expected_value[1],
    shap_values=shap_values[1][example_idx],
    features=X_test.iloc[example_idx],
    matplotlib=True,
)
plt.show()



**Комментарий к локальной интерпретации.**  
На force plot видно, какие признаки именно для выбранного пациента сдвинули прогноз в сторону наличия болезни сердца, а какие — в сторону её отсутствия. Такой анализ полезен в медицинском контексте, потому что позволяет не просто получить риск, а понять, на каких конкретно факторах модель его основывает.



## 7. Экспертная оценка интерпретации

С точки зрения предметной области результаты выглядят адекватно, если среди ключевых признаков действительно оказываются `cp`, `exang`, `oldpeak`, `thalach`, `ca` и `thal`: именно они тесно связаны с ишемией, нарушениями перфузии миокарда и результатами нагрузочных тестов. Если же модель начинает опираться в основном на второстепенные или слабо объяснимые признаки, это был бы сигнал проверить переобучение, устойчивость модели и корректность данных.



## 8. Вывод

В этом задании была построена более сложная ансамблевая модель — `RandomForestClassifier`. Для неё выполнен подбор гиперпараметров на кросс-валидации с помощью `GridSearchCV`, после чего лучшая модель оценена на отложенной выборке по метрикам **ROC-AUC** и **Recall**.

Дополнительно модель была проинтерпретирована двумя способами:
- **глобально** — через permutation importance;
- **локально** — через SHAP для отдельного пациента.

Такой подход позволяет не только повысить качество по сравнению с простым бейзлайном, но и сделать модель более прозрачной, что особенно важно в медицинских задачах.
